# APIGEE Ingestion Daily — Fixed

## Problem
The original script was getting ~5k fewer records than the Elastic portal because the
`size: 10000` cap silently truncated intervals with more than 10,000 documents.

With 914,897 records across 144 intervals the **average per interval is ~6,354**.
Peak daytime intervals routinely exceed 10,000 — Elasticsearch returns exactly 10,000
records and the remainder are dropped without any error.

## Fix
Added `search_after` pagination inside each interval loop.
If a batch returns exactly 10,000 hits we continue fetching the next page using the
sort value of the last document, until the batch is smaller than 10,000.

In [ ]:
import requests
import pytz
from datetime import datetime, timedelta

kolkata_tz = pytz.timezone("Asia/Kolkata")

user = "apigee_dbp"
psd  = "apigee@123"
url  = "https://10.227.12.188:9201/apigee_updated_solace/_search"

headers = {
    "Content-Type": "application/vnd.elasticsearch+json; compatible-with=8",
    "Accept":       "application/vnd.elasticsearch+json; compatible-with=8",
}

# Replace with your actual SCOPE_JOURNEY_MAP
SCOPE_JOURNEY_MAP = {
    "TDCC": [],
    "SNCC": [],
    "PPCC": [],
    "PTCC": [],
}
all_scopes = list(SCOPE_JOURNEY_MAP.keys())

file_date       = (datetime.now(kolkata_tz).date() - timedelta(days=1)).strftime("%Y-%m-%d")
interval_minute = 10
total_intervals = (24 * 60) // interval_minute   # 144

print(f"file_date      : {file_date}")
print(f"total_intervals: {total_intervals}")

In [ ]:
all_hits = []

for i in range(total_intervals):
    start_total_min = i * interval_minute
    start_hour      = start_total_min // 60
    start_minute    = start_total_min % 60

    # -1 so the end sits within the same minute-block; next interval starts cleanly
    end_total_min = start_total_min + interval_minute - 1
    end_hour      = end_total_min // 60
    end_minute    = end_total_min % 60

    start_time = f"{file_date}T{start_hour:02d}:{start_minute:02d}:00.000"
    end_time   = f"{file_date}T{end_hour:02d}:{end_minute:02d}:59.999"
    print(f"Fetching interval {i + 1}/{total_intervals}: {start_time} to {end_time}")

    interval_hits = []
    search_after  = None     # pagination cursor
    page          = 0

    while True:
        page += 1
        query_body = {
            "size":   10000,
            "fields": ["*"],
            # sort is REQUIRED for search_after to work correctly
            "sort": [{"@timestamp": "asc", "_id": "asc"}],
            "query": {
                "bool": {
                    "must": [
                        {"range": {"@timestamp": {"gte": start_time, "lte": end_time}}},
                        {"terms": {"Scope.keyword": all_scopes}},
                    ]
                }
            },
        }

        if search_after:
            query_body["search_after"] = search_after

        response = requests.post(
            url,
            headers=headers,
            auth=(user, psd),
            json=query_body,
            verify=False,
            timeout=120,
        )

        if response.status_code != 200:
            raise Exception(
                f"API call failed for interval {start_time} - {end_time} "
                f"(page {page}): {response.text[-200]}"
            )

        resp_json = response.json()
        hits      = resp_json.get("hits", {}).get("hits", [])
        interval_hits.extend(hits)

        if len(hits) < 10000:
            # Fewer than the page size → we have all records for this interval
            break

        # Hit the cap — get the sort values of the last document and fetch next page
        search_after = hits[-1]["sort"]
        print(f"  page {page} returned 10000 hits — fetching next page ...")

    all_hits.extend(interval_hits)
    print(
        f"interval records: {len(interval_hits)} "
        f"({'pages: ' + str(page) + ', ' if page > 1 else ''}"
        f"running total: {len(all_hits)})"
    )

print(f"\nAll {total_intervals} intervals fetched successfully. Total records: {len(all_hits)}")